# Libraries 📚

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import StratifiedGroupKFold

from pathlib import Path

# Data 💾

In [2]:
path = r"C:\Users\malik\Documents\AI\my_prjects\fetal_anomalies\fetal_vision\data\raw\FETAL_PLANES_DB_data.csv"
df_original = pd.read_csv(path, sep=";")

df_original.head()

,Image_name,Patient_num,Plane,Brain_plane,Operator,US_Machine,Train
0,Patient00001_Plane1_1_of_15,1,Other,Not A Brain,Other,Aloka,1
1,Patient00001_Plane1_2_of_15,1,Other,Not A Brain,Other,Aloka,1
2,Patient00001_Plane1_3_of_15,1,Other,Not A Brain,Other,Aloka,1
3,Patient00001_Plane1_4_of_15,1,Other,Not A Brain,Other,Aloka,1
4,Patient00001_Plane1_5_of_15,1,Other,Not A Brain,Other,Aloka,1


In [3]:
df = df_original.copy()

In [4]:
df.rename(columns={"Train ": "Train"}, inplace=True)

In [5]:
df["Train"].value_counts()

Train
1    7129
0    5271
Name: count, dtype: int64

In [6]:
fig = px.pie(df, names="Train", title="Train / Test Distribution")
fig.show()

In [7]:
## We have "Train " column that does the split according to the article
# quote: "The benchmark protocol was designed to mimic a real scenario: 
# images were separated by patient and study date, using the first half of the patients to train and the next half for testing"
# Official split:
# - 50% / 50% of patients
# - approximately 53% / 47% of images

# Let's make sure there is no data leakage
train_patients = set(df[df["Train"] == 1]["Patient_num"])
test_patients  = set(df[df["Train"] == 0]["Patient_num"])

intersection = train_patients.intersection(test_patients)

print("Overlap patients:", len(intersection))
if len(intersection)> 0:
    print(intersection)

Overlap patients: 0


In [8]:
print("Train patients:", len(train_patients))
print("Test patients:", len(test_patients))

Train patients: 896
Test patients: 896


In [9]:
patient_split_check = df.groupby("Patient_num")["Train"].nunique()

print(patient_split_check.value_counts())

Train
1    1792
Name: count, dtype: int64


In [10]:
all_patients = set(df_original["Patient_num"].unique())
print("Patients that has no groups:", len(all_patients - train_patients - test_patients))

Patients that has no groups: 0


# Split🪓

In [11]:
# Train / Test Split
offi_train_df = df[df["Train"]==1].copy()

test_df = df[df["Train"]==0].copy()

# 20% Validation set from Train set
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=9)

train_idx, val_idx = next(
    sgkf.split(
        X= offi_train_df,
        y= offi_train_df["Plane"],
        groups=offi_train_df["Patient_num"]
    )
)
train_df = offi_train_df.iloc[train_idx].copy()
val_df = offi_train_df.iloc[val_idx].copy()



In [12]:
train_idx.shape
val_idx.shape

(1427,)

In [13]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (5702, 7)
Validation: (1427, 7)
Test: (5271, 7)


In [14]:
for i, (train_index, val_index) in enumerate(
    sgkf.split(
        X=offi_train_df,
        y=offi_train_df["Plane"],
        groups=offi_train_df["Patient_num"]
    ),
    start=1
):
    fold_train = offi_train_df.iloc[train_index]
    fold_val = offi_train_df.iloc[val_index]

    train_patients = set(fold_train["Patient_num"])
    val_patients = set(fold_val["Patient_num"])

    print(f"\n --- Fold {i} ---")
    print(f"Train: {len(fold_train)} images | {len(train_patients)} patients")
    print(f"Validation: {len(fold_val)} images | {len(val_patients)} patients")
    print("Patient overlap:", len(train_patients & val_patients))


 --- Fold 1 ---
Train: 5702 images | 724 patients
Validation: 1427 images | 172 patients
Patient overlap: 0

 --- Fold 2 ---
Train: 5705 images | 713 patients
Validation: 1424 images | 183 patients
Patient overlap: 0

 --- Fold 3 ---
Train: 5707 images | 712 patients
Validation: 1422 images | 184 patients
Patient overlap: 0

 --- Fold 4 ---
Train: 5703 images | 720 patients
Validation: 1426 images | 176 patients
Patient overlap: 0

 --- Fold 5 ---
Train: 5699 images | 715 patients
Validation: 1430 images | 181 patients
Patient overlap: 0


In [15]:
train_patients = set(train_df["Patient_num"])
val_patients = set(val_df["Patient_num"])
test_patients = set(test_df["Patient_num"])

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

print("\nPatient overlap:")
print("Train / Validation:", len(train_patients & val_patients))
print("Train / Test:", len(train_patients & test_patients))
print("Validation / Test:", len(val_patients & test_patients))

Train patients: 724
Validation patients: 172
Test patients: 896

Patient overlap:
Train / Validation: 0
Train / Test: 0
Validation / Test: 0


In [17]:
split_dir = Path(
    r"C:\Users\malik\Documents\AI\my_prjects\fetal_anomalies\fetal_vision\data\splits"
)

train_df.to_csv(split_dir / "train.csv", index=False)
val_df.to_csv(split_dir / "val.csv", index=False)
test_df.to_csv(split_dir / "test.csv", index=False)